In [15]:
# ==============================================================
# FLUXO DE POTÊNCIA - MÉTODO DE NEWTON-RAPHSON
# ==============================================================

import numpy as np
import pandas as pd


# ==============================================================
# CONFIGURAÇÕES GERAIS E LIMITES (EC2)
# ==============================================================

ARQUIVO_ENTRADA = 'dados_sistema13B_EC2.txt'

S_BASE = 100.0          # MVA
TOLERANCIA = 1e-6
MAX_ITERACOES = 20

# --------------------------------------------------------------
# Limites Operativos definidos para o Estudo de Caso 2
# --------------------------------------------------------------
Q_GEN_MAX = 90.0        # MVAr
Q_GEN_MIN = -70.0       # MVAr
V_MAX = 1.05            # pu (ONS - Submódulo 2.3)
V_MIN = 0.95            # pu (ONS - Submódulo 2.3)
TAP_MAX = 1.1           # pu
TAP_MIN = 0.9           # pu
DEF_MAX = 30.0          # graus
DEF_MIN = -30.0         # graus

# True  -> mostra detalhes de cada iteração
# False -> mostra apenas o resumo e os resultados finais
MOSTRAR_ITERACOES = True

np.set_printoptions(
    precision=6,
    suppress=True,
    linewidth=150
)


# ==============================================================
# FUNÇÕES DE APRESENTAÇÃO
# ==============================================================

def titulo(texto, largura=80):
    """Imprime um título padronizado."""
    print()
    print("=" * largura)
    print(texto.center(largura))
    print("=" * largura)


def subtitulo(texto, largura=80):
    """Imprime um subtítulo padronizado."""
    print()
    print("-" * largura)
    print(texto)
    print("-" * largura)


def imprimir_dataframe(df, casas=4):
    """Imprime DataFrame com formatação numérica padronizada."""

    formatadores = {}

    for coluna in df.columns:
        if pd.api.types.is_float_dtype(df[coluna]):
            formatadores[coluna] = (
                lambda x, c=casas: f"{x:.{c}f}"
            )

    print(
        df.to_string(
            index=False,
            formatters=formatadores
        )
    )


def imprimir_matriz(matriz, nome, barras=None):
    """Apresenta uma matriz de forma organizada usando DataFrame."""

    if barras is None:
        barras = range(1, matriz.shape[0] + 1)

    nomes = [f"B{b}" for b in barras]

    df = pd.DataFrame(
        matriz,
        index=nomes,
        columns=nomes
    )

    print(f"\n{nome}:")
    print(df.to_string(float_format=lambda x: f"{x:10.6f}"))


# ==============================================================
# PASSO 1 - LEITURA DOS DADOS
# ==============================================================

def localizar_blocos(arquivo):
    """
    Localiza os blocos DBAR e DCIR no arquivo de entrada.
    """

    with open(arquivo, 'r', encoding='latin-1') as f:
        linhas = f.readlines()

    linha_dbar_inicio = None
    linha_dbar_fim = None

    linha_dcir_inicio = None
    linha_dcir_fim = None

    bloco_atual = None

    for i, linha in enumerate(linhas):

        texto = linha.strip()

        if texto == 'DBAR':
            linha_dbar_inicio = i + 2
            bloco_atual = 'DBAR'

        elif texto == 'DCIR':
            linha_dcir_inicio = i + 2
            bloco_atual = 'DCIR'

        elif '####' in texto:

            if bloco_atual == 'DBAR':
                linha_dbar_fim = i
                bloco_atual = None

            elif bloco_atual == 'DCIR':
                linha_dcir_fim = i
                bloco_atual = None

    return (
        linha_dbar_inicio,
        linha_dbar_fim,
        linha_dcir_inicio,
        linha_dcir_fim
    )


def ler_dados(arquivo):
    """
    Realiza a leitura das tabelas DBAR e DCIR.
    """

    (
        dbar_inicio,
        dbar_fim,
        dcir_inicio,
        dcir_fim
    ) = localizar_blocos(arquivo)

    if None in [dbar_inicio, dbar_fim, dcir_inicio, dcir_fim]:
        raise ValueError(
            "Não foi possível localizar corretamente "
            "os blocos DBAR e DCIR."
        )

    n_dbar = dbar_fim - dbar_inicio - 1
    n_dcir = dcir_fim - dcir_inicio - 1

    # ----------------------------------------------------------
    # DBAR
    # ----------------------------------------------------------

    dbar = pd.read_csv(
        arquivo,
        sep=r'\s+',
        skiprows=dbar_inicio,
        nrows=n_dbar,
        engine='python',
        encoding='latin-1'
    )

    dbar.columns = dbar.columns.str.strip()

    # Padronização da coluna de ângulo
    coluna_angulo = [
        coluna
        for coluna in dbar.columns
        if 'Oesp' in coluna
    ]

    if coluna_angulo:
        dbar = dbar.rename(
            columns={
                coluna_angulo[0]: 'Oesp(graus)'
            }
        )

    # Garante índices sequenciais internos
    dbar = dbar.reset_index(drop=True)

    # ----------------------------------------------------------
    # DCIR
    # ----------------------------------------------------------

    dcir = pd.read_csv(
        arquivo,
        sep=r'\s+',
        skiprows=dcir_inicio,
        nrows=n_dcir,
        engine='python',
        encoding='latin-1'
    )

    dcir.columns = dcir.columns.str.strip()

    return dbar, dcir


# ==============================================================
# PASSO 2 - MATRIZ YBUS
# ==============================================================

def montar_ybus(dbar, dcir):
    """
    Monta a matriz de admitância nodal Ybus.
    """

    n_barras = len(dbar)

    ybus = np.zeros(
        (n_barras, n_barras),
        dtype=complex
    )

    # ----------------------------------------------------------
    # Mapeamento:
    # número da barra -> posição dentro das matrizes
    # ----------------------------------------------------------

    mapa_barras = {
        int(numero): indice
        for indice, numero in enumerate(dbar['BARRA'])
    }

    # ----------------------------------------------------------
    # Circuitos
    # ----------------------------------------------------------

    for _, ramo in dcir.iterrows():

        barra_de = int(ramo['BDE'])
        barra_para = int(ramo['BPARA'])

        i = mapa_barras[barra_de]
        j = mapa_barras[barra_para]

        R = float(ramo['RES(PU)'])
        X = float(ramo['REAT(PU)'])
        b = float(ramo['SUCsh(PU)'])

        # Tap
        tap = ramo['TAP(PU)']

        if pd.isna(tap) or tap == 0:
            tap = 1.0

        # Defasagem
        defasagem = ramo['DEF(GRAUS)']

        if pd.isna(defasagem):
            defasagem = 0.0

        # Convenção ANAREDE
        def_rad = -np.deg2rad(defasagem)

        a = tap * np.exp(1j * def_rad)

        # Admitância série
        y = 1.0 / (R + 1j * X)

        # Susceptância shunt total
        b_shunt = 1j * b

        # Modelo π generalizado
        ybus[i, i] += (
            y / abs(a)**2
            + b_shunt / 2
        )

        ybus[j, j] += (
            y
            + b_shunt / 2
        )

        ybus[i, j] -= y / np.conj(a)
        ybus[j, i] -= y / a

    # ----------------------------------------------------------
    # Shunts das barras
    # ----------------------------------------------------------

    for indice, barra in dbar.iterrows():

        q_shunt = float(barra['Bsh(PU)'])

        if q_shunt != 0:
            ybus[indice, indice] += 1j * q_shunt

    return ybus, mapa_barras


# ==============================================================
# PASSO 3 - CLASSIFICAÇÃO DAS BARRAS
# ==============================================================

def classificar_barras(dbar):
    """
    Identifica barras SW, PV e PQ e monta as potências
    especificadas.
    """

    sw = dbar.index[
        dbar['TIPO'] == 'SW'
    ].tolist()

    pv = dbar.index[
        dbar['TIPO'] == 'PV'
    ].tolist()

    pq = dbar.index[
        dbar['TIPO'] == 'PQ'
    ].tolist()

    if not sw:
        raise ValueError("Nenhuma barra SW encontrada.")

    # ----------------------------------------------------------
    # Potências especificadas
    # ----------------------------------------------------------

    P_spec = (
        dbar['PGesp(PU)']
        - dbar['PD(PU)']
    ).to_numpy(dtype=float)

    Q_spec = (
        -dbar['QD(PU)']
    ).to_numpy(dtype=float)

    # P -> PV + PQ
    indices_P = sorted(pv + pq)

    # Q -> somente PQ
    indices_Q = sorted(pq)

    return {
        'SW': sw,
        'PV': pv,
        'PQ': pq,
        'indices_P': indices_P,
        'indices_Q': indices_Q,
        'P_spec': P_spec,
        'Q_spec': Q_spec
    }


# ==============================================================
# CÁLCULO DE P E Q
# ==============================================================

def calcular_potencias(V, theta, G, B):
    """
    Calcula as injeções de potência ativa e reativa
    nas barras.
    """

    n = len(V)

    P = np.zeros(n)
    Q = np.zeros(n)

    for k in range(n):

        soma_P = 0.0
        soma_Q = 0.0

        for m in range(n):

            theta_km = theta[k] - theta[m]

            soma_P += V[m] * (
                G[k, m] * np.cos(theta_km)
                + B[k, m] * np.sin(theta_km)
            )

            soma_Q += V[m] * (
                G[k, m] * np.sin(theta_km)
                - B[k, m] * np.cos(theta_km)
            )

        P[k] = V[k] * soma_P
        Q[k] = V[k] * soma_Q

    return P, Q


# ==============================================================
# PASSO 5 - JACOBIANA
# ==============================================================

def montar_jacobiana(
    V,
    theta,
    G,
    B,
    P,
    Q,
    indices_P,
    indices_Q
):
    """
    Monta as submatrizes H, N, M e L
    da Jacobiana do Newton-Raphson.
    """

    n = len(V)

    H = np.zeros((n, n))
    N = np.zeros((n, n))
    M = np.zeros((n, n))
    L = np.zeros((n, n))

    for k in range(n):

        for m in range(n):

            theta_km = theta[k] - theta[m]

            if k == m:

                H[k, k] = (
                    -V[k]**2 * B[k, k]
                    - Q[k]
                )

                N[k, k] = (
                    V[k]**2 * G[k, k]
                    + P[k]
                ) / V[k]

                M[k, k] = (
                    -V[k]**2 * G[k, k]
                    + P[k]
                )

                L[k, k] = (
                    -V[k]**2 * B[k, k]
                    + Q[k]
                ) / V[k]

            else:

                H[k, m] = (
                    V[k]
                    * V[m]
                    * (
                        G[k, m] * np.sin(theta_km)
                        - B[k, m] * np.cos(theta_km)
                    )
                )

                N[k, m] = (
                    V[k]
                    * (
                        G[k, m] * np.cos(theta_km)
                        + B[k, m] * np.sin(theta_km)
                    )
                )

                M[k, m] = (
                    -V[k]
                    * V[m]
                    * (
                        G[k, m] * np.cos(theta_km)
                        + B[k, m] * np.sin(theta_km)
                    )
                )

                L[k, m] = (
                    V[k]
                    * (
                        G[k, m] * np.sin(theta_km)
                        - B[k, m] * np.cos(theta_km)
                    )
                )

    # ----------------------------------------------------------
    # Redução da Jacobiana
    # ----------------------------------------------------------

    Hr = H[np.ix_(indices_P, indices_P)]
    Nr = N[np.ix_(indices_P, indices_Q)]

    Mr = M[np.ix_(indices_Q, indices_P)]
    Lr = L[np.ix_(indices_Q, indices_Q)]

    J = np.block([
        [Hr, Nr],
        [Mr, Lr]
    ])

    return J, Hr, Nr, Mr, Lr


# ==============================================================
# PASSO 4 A 7 - NEWTON-RAPHSON
# ==============================================================

def newton_raphson(
    dbar,
    ybus,
    dados_barras,
    tolerancia=TOLERANCIA,
    max_iteracoes=MAX_ITERACOES
):
    """
    Resolve o fluxo de potência pelo método
    de Newton-Raphson.
    """

    G = ybus.real
    B = ybus.imag

    indices_P = dados_barras['indices_P']
    indices_Q = dados_barras['indices_Q']

    P_spec = dados_barras['P_spec']
    Q_spec = dados_barras['Q_spec']

    # ----------------------------------------------------------
    # Condições iniciais
    # ----------------------------------------------------------

    V = dbar['Vesp(PU)'].to_numpy(dtype=float)

    theta = np.deg2rad(
        dbar['Oesp(graus)'].to_numpy(dtype=float)
    )

    historico = []

    convergiu = False

    # ----------------------------------------------------------
    # Processo iterativo
    # ----------------------------------------------------------

    for iteracao in range(1, max_iteracoes + 1):

        # ------------------------------------------------------
        # Potências calculadas
        # ------------------------------------------------------

        P_calc, Q_calc = calcular_potencias(
            V,
            theta,
            G,
            B
        )

        # ------------------------------------------------------
        # Mismatches
        # ------------------------------------------------------

        delta_P = P_spec - P_calc
        delta_Q = Q_spec - Q_calc

        g = np.concatenate([
            delta_P[indices_P],
            delta_Q[indices_Q]
        ])

        erro_max = np.max(np.abs(g))

        historico.append({
            'Iteração': iteracao,
            'Erro Máximo': erro_max
        })

        # ------------------------------------------------------
        # Saída da iteração
        # ------------------------------------------------------

        if MOSTRAR_ITERACOES:

            subtitulo(
                f"ITERAÇÃO {iteracao}"
            )

            print(
                f"Erro máximo dos mismatches: "
                f"{erro_max:.6e}"
            )

            print(
                "\nV (pu):      ",
                np.round(V, 6)
            )

            print(
                "θ (graus):   ",
                np.round(
                    np.rad2deg(theta),
                    6
                )
            )

            print(
                "ΔP / ΔQ:     ",
                np.round(g, 8)
            )

        # ------------------------------------------------------
        # Convergência
        # ------------------------------------------------------

        if erro_max < tolerancia:

            convergiu = True
            break

        # ------------------------------------------------------
        # Jacobiana
        # ------------------------------------------------------

        J, H, N, M, L = montar_jacobiana(
            V,
            theta,
            G,
            B,
            P_calc,
            Q_calc,
            indices_P,
            indices_Q
        )

        if MOSTRAR_ITERACOES:

            print("\nJacobiana:")
            print(
                pd.DataFrame(J).to_string(
                    index=False,
                    header=False,
                    float_format=lambda x: f"{x:11.6f}"
                )
            )

        # ------------------------------------------------------
        # Sistema linear
        #
        # J Δx = g
        # ------------------------------------------------------

        try:

            delta_x = np.linalg.solve(
                J,
                g
            )

        except np.linalg.LinAlgError:

            raise RuntimeError(
                "A matriz Jacobiana é singular. "
                "Não foi possível continuar."
            )

        # ------------------------------------------------------
        # Correções
        # ------------------------------------------------------

        n_angulos = len(indices_P)

        delta_theta = delta_x[:n_angulos]
        delta_V = delta_x[n_angulos:]

        # ------------------------------------------------------
        # Atualização
        # ------------------------------------------------------

        theta[indices_P] += delta_theta
        V[indices_Q] += delta_V

    # ----------------------------------------------------------
    # Recalcula P e Q no estado final
    # ----------------------------------------------------------

    P_calc, Q_calc = calcular_potencias(
        V,
        theta,
        G,
        B
    )

    return {
        'V': V,
        'theta': theta,
        'P': P_calc,
        'Q': Q_calc,
        'convergiu': convergiu,
        'iteracoes': iteracao,
        'erro': erro_max,
        'historico': pd.DataFrame(historico)
    }


# ==============================================================
# PASSO 8 - RESULTADOS DAS BARRAS
# ==============================================================

def resultados_barras(
    dbar,
    resultado_nr,
    s_base=S_BASE
):
    """
    Monta a tabela final das barras.
    """

    V = resultado_nr['V']
    theta = resultado_nr['theta']

    P = resultado_nr['P']
    Q = resultado_nr['Q']

    Pd = dbar['PD(PU)'].to_numpy(dtype=float)
    Qd = dbar['QD(PU)'].to_numpy(dtype=float)

    # ----------------------------------------------------------
    # Geração final
    # ----------------------------------------------------------

    Pg = P + Pd
    Qg = Q + Qd

    resultados = pd.DataFrame({

        'Barra':
            dbar['BARRA'],

        'Tipo':
            dbar['TIPO'],

        'V (pu)':
            V,

        'Ângulo (°)':
            np.rad2deg(theta),

        'PG (MW)':
            Pg * s_base,

        'QG (MVAr)':
            Qg * s_base,

        'PD (MW)':
            Pd * s_base,

        'QD (MVAr)':
            Qd * s_base,

        'Bsh (MVAr)':
            dbar['Bsh(PU)'].to_numpy(dtype=float)
            * s_base
    })

    return resultados


# ==============================================================
# PASSO 9 - FLUXOS NOS CIRCUITOS
# ==============================================================

def calcular_fluxos(
    dcir,
    mapa_barras,
    V,
    theta,
    s_base=S_BASE
):
    """
    Calcula os fluxos de potência nos dois sentidos
    de cada circuito.
    """

    dados_fluxos = []

    for _, linha in dcir.iterrows():

        barra_k = int(linha['BDE'])
        barra_m = int(linha['BPARA'])

        k = mapa_barras[barra_k]
        m = mapa_barras[barra_m]

        R = float(linha['RES(PU)'])
        X = float(linha['REAT(PU)'])
        b = float(linha['SUCsh(PU)'])

        # ------------------------------------------------------
        # Tap
        # ------------------------------------------------------

        tap = linha['TAP(PU)']

        if pd.isna(tap) or tap == 0:
            tap = 1.0

        # ------------------------------------------------------
        # Defasagem
        # ------------------------------------------------------

        defasagem = linha['DEF(GRAUS)']

        if pd.isna(defasagem):
            defasagem = 0.0

        def_rad = -np.deg2rad(defasagem)

        a = tap * np.exp(
            1j * def_rad
        )

        # ------------------------------------------------------
        # Admitâncias
        # ------------------------------------------------------

        y = 1.0 / (
            R + 1j * X
        )

        y_sh = 1j * b / 2

        # ------------------------------------------------------
        # Tensões complexas
        # ------------------------------------------------------

        Vk = V[k] * np.exp(
            1j * theta[k]
        )

        Vm = V[m] * np.exp(
            1j * theta[m]
        )

        # ------------------------------------------------------
        # Correntes
        # ------------------------------------------------------

        I_km = (
            Vk * y / abs(a)**2
            + Vk * y_sh
            - Vm * y / np.conj(a)
        )

        I_mk = (
            Vm * y
            + Vm * y_sh
            - Vk * y / a
        )

        # ------------------------------------------------------
        # Potências complexas
        # ------------------------------------------------------

        S_km = Vk * np.conj(I_km)
        S_mk = Vm * np.conj(I_mk)

        dados_fluxos.append({

            'De': barra_k,
            'Para': barra_m,

            'Pkm (MW)':
                S_km.real * s_base,

            'Qkm (MVAr)':
                S_km.imag * s_base,

            'Skm (MVA)':
                abs(S_km) * s_base,

            'Pmk (MW)':
                S_mk.real * s_base,

            'Qmk (MVAr)':
                S_mk.imag * s_base,

            'Smk (MVA)':
                abs(S_mk) * s_base,

            'Perda P (MW)':
                (
                    S_km.real
                    + S_mk.real
                ) * s_base,

            'Perda Q (MVAr)':
                (
                    S_km.imag
                    + S_mk.imag
                ) * s_base
        })

    return pd.DataFrame(dados_fluxos)


# ==============================================================
# PASSO 10 - CARREGAMENTO DOS CIRCUITOS
# ==============================================================

def verificar_capacidade(
    dcir,
    fluxos,
    s_base=S_BASE
):
    """
    Verifica a capacidade e o carregamento
    de cada circuito.
    """

    resultado = fluxos.copy()

    if 'CAP(PU)' in dcir.columns:

        capacidade = (
            dcir['CAP(PU)']
            .fillna(1.0)
            .to_numpy(dtype=float)
            * s_base
        )

    else:

        capacidade = np.full(
            len(resultado),
            s_base
        )

    resultado['Capacidade (MVA)'] = capacidade

    resultado['Smax (MVA)'] = resultado[
        ['Skm (MVA)', 'Smk (MVA)']
    ].max(axis=1)

    resultado['Carregamento (%)'] = (
        resultado['Smax (MVA)']
        / resultado['Capacidade (MVA)']
        * 100
    )

    resultado['Status'] = np.where(
        resultado['Carregamento (%)'] > 100,
        'EXCEDEU',
        'OK'
    )

    return resultado


# ==============================================================
# RELATÓRIO FINAL ORIGINAL
# ==============================================================

def apresentar_resultados(
    dbar,
    dcir,
    ybus,
    resultado_nr,
    resultados_barras_df,
    fluxos_df,
    capacidade_df
):
    """
    Apresentação organizada de todos os resultados.
    """

    # ----------------------------------------------------------
    # RESUMO DA SOLUÇÃO
    # ----------------------------------------------------------

    titulo(
        "FLUXO DE POTÊNCIA - NEWTON-RAPHSON"
    )

    if resultado_nr['convergiu']:

        print(
            f"\nStatus             : CONVERGIU"
        )

    else:

        print(
            f"\nStatus             : NÃO CONVERGIU"
        )

    print(
        f"Iterações          : "
        f"{resultado_nr['iteracoes']}"
    )

    print(
        f"Erro máximo final  : "
        f"{resultado_nr['erro']:.6e}"
    )

    print(
        f"Tolerância         : "
        f"{TOLERANCIA:.2e}"
    )

    print(
        f"Base de potência   : "
        f"{S_BASE:.2f} MVA"
    )

    # ----------------------------------------------------------
    # RESULTADOS DAS BARRAS
    # ----------------------------------------------------------

    titulo(
        "RESULTADOS DAS BARRAS"
    )

    imprimir_dataframe(
        resultados_barras_df,
        casas=4
    )

    # ----------------------------------------------------------
    # FLUXOS
    # ----------------------------------------------------------

    titulo(
        "FLUXOS DE POTÊNCIA NOS CIRCUITOS"
    )

    colunas_fluxo = [

        'De',
        'Para',

        'Pkm (MW)',
        'Qkm (MVAr)',
        'Skm (MVA)',

        'Pmk (MW)',
        'Qmk (MVAr)',
        'Smk (MVA)',

        'Perda P (MW)',
        'Perda Q (MVAr)'
    ]

    imprimir_dataframe(
        fluxos_df[colunas_fluxo],
        casas=3
    )

    # ----------------------------------------------------------
    # PERDAS
    # ----------------------------------------------------------

    perda_P = fluxos_df[
        'Perda P (MW)'
    ].sum()

    perda_Q = fluxos_df[
        'Perda Q (MVAr)'
    ].sum()

    titulo(
        "PERDAS TOTAIS DO SISTEMA"
    )

    print(
        f"Perdas ativas   : "
        f"{perda_P:.4f} MW"
    )

    print(
        f"Perdas reativas : "
        f"{perda_Q:.4f} MVAr"
    )

    # ----------------------------------------------------------
    # CAPACIDADES
    # ----------------------------------------------------------

    titulo(
        "CARREGAMENTO DOS CIRCUITOS"
    )

    colunas_capacidade = [

        'De',
        'Para',
        'Smax (MVA)',
        'Capacidade (MVA)',
        'Carregamento (%)',
        'Status'
    ]

    imprimir_dataframe(
        capacidade_df[colunas_capacidade],
        casas=2
    )

    # ----------------------------------------------------------
    # SOBRECARGAS
    # ----------------------------------------------------------

    sobrecargas = capacidade_df[
        capacidade_df['Status'] == 'EXCEDEU'
    ]

    print()

    if sobrecargas.empty:

        print(
            "Situação: todas as linhas estão "
            "dentro dos limites de capacidade."
        )

    else:

        print(
            "ATENÇÃO: foram encontradas "
            "linhas sobrecarregadas:"
        )

        for _, linha in sobrecargas.iterrows():

            print(
                f"  Linha "
                f"{int(linha['De'])} -> "
                f"{int(linha['Para'])}: "
                f"{linha['Carregamento (%)']:.2f}%"
            )


# ==============================================================
# PASSO 11 - DIAGNÓSTICO DO ESTUDO DE CASO 2 (NOVO BLOCO)
# ==============================================================

def relatorio_diagnostico_ec2(
    barras_df,
    capacidade_df,
    dcir
):
    """
    Função dedicada para atender às restrições do Estudo de Caso 2.
    Faz a varredura nos resultados em busca de violações.
    """

    titulo("DIAGNÓSTICO TÉCNICO DOS LIMITES - ESTUDO DE CASO 2")

    alertas = 0

    # ----------------------------------------------------------
    # 1. Verificação das Tensões nas Barras
    # ----------------------------------------------------------

    print("\n[ TENSÃO NAS BARRAS ]")

    tensoes_fora = barras_df[
        (barras_df['V (pu)'] < V_MIN) |
        (barras_df['V (pu)'] > V_MAX)
    ]

    if tensoes_fora.empty:
        print(f"  OK: Todas as tensões estão dentro da faixa do ONS ({V_MIN} a {V_MAX} pu).")
    else:
        for _, b in tensoes_fora.iterrows():
            print(f"  ALERTA: Barra {b['Barra']} violou a tensão limite operando em {b['V (pu)']:.4f} pu.")
            alertas += 1

    # ----------------------------------------------------------
    # 2. Verificação dos Limites de Geração Reativa
    # ----------------------------------------------------------

    print(f"\n[ LIMITES DE POTÊNCIA REATIVA ({Q_GEN_MIN} a {Q_GEN_MAX} MVAr) ]")

    geradores = barras_df[
        barras_df['Tipo'].isin(['PV', 'SW'])
    ]

    q_violado = geradores[
        (geradores['QG (MVAr)'] < Q_GEN_MIN) |
        (geradores['QG (MVAr)'] > Q_GEN_MAX)
    ]

    if q_violado.empty:
        print("  OK: Nenhum gerador violou sua capacidade de potência reativa.")
    else:
        for _, g in q_violado.iterrows():
            print(f"  ALERTA: Unidade geradora na Barra {g['Barra']} excedeu o limite injetando {g['QG (MVAr)']:.2f} MVAr.")
            alertas += 1

    # ----------------------------------------------------------
    # 3. Verificação dos Transformadores (Taps e Defasagem)
    # ----------------------------------------------------------

    print("\n[ LIMITES DE CONTROLE DOS TRANSFORMADORES ]")

    transf = dcir[
        dcir['TAP(PU)'].notna() |
        dcir['DEF(GRAUS)'].notna()
    ]

    transf_violados = False

    for _, t in transf.iterrows():

        tap = float(t['TAP(PU)']) if pd.notna(t['TAP(PU)']) else 1.0
        defas = float(t['DEF(GRAUS)']) if pd.notna(t['DEF(GRAUS)']) else 0.0

        if tap != 1.0 and not (TAP_MIN <= tap <= TAP_MAX):
            print(f"  ALERTA: Trafo {int(t['BDE'])}-{int(t['BPARA'])} com tap {tap} pu (limites: {TAP_MIN} a {TAP_MAX} pu).")
            transf_violados = True
            alertas += 1

        if defas != 0.0 and not (DEF_MIN <= defas <= DEF_MAX):
            print(f"  ALERTA: Trafo {int(t['BDE'])}-{int(t['BPARA'])} com ângulo {defas}° (limites: {DEF_MIN} a {DEF_MAX} graus).")
            transf_violados = True
            alertas += 1

    if not transf_violados:
        print("  OK: Todos os ajustes de tap e de defasagem angular respeitam as restrições físicas.")

    # ----------------------------------------------------------
    # 4. Verificação de Sobrecarga nos Circuitos
    # ----------------------------------------------------------

    print("\n[ CARREGAMENTO DOS CIRCUITOS ]")

    sobrecargas = capacidade_df[
        capacidade_df['Status'] == 'EXCEDEU'
    ]

    if sobrecargas.empty:
        print("  OK: Nenhum circuito ultrapassou 100% da sua capacidade máxima de MVA.")
    else:
        for _, linha in sobrecargas.iterrows():
            print(f"  ALERTA: Linha {int(linha['De'])}-{int(linha['Para'])} sobrecarregada em {linha['Carregamento (%)']:.2f}%.")
            alertas += 1

    # ----------------------------------------------------------
    # Conclusão
    # ----------------------------------------------------------

    if alertas == 0:
        print("\n>>> CONCLUSÃO: O cenário de operação redespachado é VIÁVEL e seguro. Nenhuma restrição do sistema foi violada. <<<")
    else:
        print(f"\n>>> CONCLUSÃO: O sistema é INVIÁVEL. Foram detectadas {alertas} restrições violadas precisando de ajuste. <<<")


# ==============================================================
# PROGRAMA PRINCIPAL
# ==============================================================

def main():

    titulo(
        "ANÁLISE DE SISTEMAS ELÉTRICOS DE POTÊNCIA"
    )

    print(
        "\nMétodo: Newton-Raphson"
    )

    print(
        f"Arquivo de entrada: {ARQUIVO_ENTRADA}"
    )

    # ==========================================================
    # 1. LEITURA
    # ==========================================================

    dbar, dcir = ler_dados(
        ARQUIVO_ENTRADA
    )

    titulo(
        "DADOS DE ENTRADA - BARRAS"
    )

    imprimir_dataframe(
        dbar,
        casas=4
    )

    titulo(
        "DADOS DE ENTRADA - CIRCUITOS"
    )

    imprimir_dataframe(
        dcir,
        casas=4
    )

    # ==========================================================
    # 2. YBUS
    # ==========================================================

    ybus, mapa_barras = montar_ybus(
        dbar,
        dcir
    )

    Gbus = ybus.real
    Bbus = ybus.imag

    # ==========================================================
    # 3. TIPOS DE BARRA
    # ==========================================================

    dados_barras = classificar_barras(
        dbar
    )

    # ==========================================================
    # 4. NEWTON-RAPHSON
    # ==========================================================

    resultado_nr = newton_raphson(
        dbar,
        ybus,
        dados_barras
    )

    # ==========================================================
    # 5. RESULTADOS DAS BARRAS
    # ==========================================================

    barras_df = resultados_barras(
        dbar,
        resultado_nr
    )

    # ==========================================================
    # 6. FLUXOS
    # ==========================================================

    fluxos_df = calcular_fluxos(
        dcir,
        mapa_barras,
        resultado_nr['V'],
        resultado_nr['theta']
    )

    # ==========================================================
    # 7. CAPACIDADE
    # ==========================================================

    capacidade_df = verificar_capacidade(
        dcir,
        fluxos_df
    )

    # ==========================================================
    # 8. RELATÓRIO FINAL
    # ==========================================================

    apresentar_resultados(
        dbar,
        dcir,
        ybus,
        resultado_nr,
        barras_df,
        fluxos_df,
        capacidade_df
    )

    # ==========================================================
    # 9. IMPRIME O DIAGNÓSTICO DO ESTUDO DE CASO 2
    # ==========================================================

    if resultado_nr['convergiu']:
        relatorio_diagnostico_ec2(
            barras_df,
            capacidade_df,
            dcir
        )
    else:
        print(
            "\n[!] O diagnóstico de limites não pode ser "
            "emitido porque o sistema não convergiu."
        )


# ==============================================================
# EXECUÇÃO
# ==============================================================

if __name__ == '__main__':
    main()


                   ANÁLISE DE SISTEMAS ELÉTRICOS DE POTÊNCIA                    

Método: Newton-Raphson
Arquivo de entrada: dados_sistema13B_EC2.txt

                           DADOS DE ENTRADA - BARRAS                            
 BARRA PD(PU) QD(PU) Bsh(PU) TIPO Vesp(PU) Oesp(graus) PGesp(PU) Cus($/MW) CGmin(PU) CGmax(PU)
     1 0.0000 0.0000  0.0000   PV   1.0000      0.0000    0.9000    0.0000    0.0000    1.2000
     2 0.7552 0.0532  0.0000   PQ   1.0000      0.0000    0.0000    0.0000    0.0000    0.0000
     3 0.7251 0.0312  0.0000   SW   1.0000      0.0000    0.0000    0.0000    0.0000    1.1000
     4 0.8922 0.0421  0.0000   PV   1.0000      0.0000    1.0000    0.0000    0.0000    1.0000
     5 0.0000 0.0000  0.0000   PQ   1.0000      0.0000    0.0000    0.0000    0.0000    0.0000
     6 0.0000 0.0000  0.0000   PQ   1.0000      0.0000    0.0000    0.0000    0.0000    0.0000
     7 0.0000 0.0000  0.0000   PV   1.0000      0.0000    0.6000    0.0000    0.0000    0.8000
     8 